## Install Required Packages

In [34]:
! pip install sacrebleu rouge_score transformers langgraph langchain chromadb langchain rapidfuzz langchain_openai python-dotenv typing ragas


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


## Setup Evaluation Metrices: [LLM_Metrics](https://github.com/Se00n00/ResearchGemma-RAG/blob/main/src/evaluation_metrics/llm_metrics.py) [Non_LLM_Metrics](https://github.com/Se00n00/ResearchGemma-RAG/blob/main/src/evaluation_metrics/non_llm_metrics.py)

In [ ]:
import os
os.environ['LLM'] = "moonshotai/kimi-k2-instruct-0905"
os.environ['BASE_URL'] = "https://api.groq.com/openai/v1"
os.environ['GROQ_API_KEY'] = ""

**Setup Judge**

In [36]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from typing import TypedDict, Annotated

class MetricScore(TypedDict):
    score: Annotated[float, ..., "Score: <0 - 1>"]

LLM = os.getenv("LLM")
BASE_URL = os.getenv("BASE_URL")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

judge = ChatOpenAI(
    model = LLM,
    api_key = GROQ_API_KEY,
    base_url = BASE_URL,
    streaming = True
).with_structured_output(MetricScore, method="json_schema", strict=True)

**LLM Evaluation Metrics**

In [37]:
correctness_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are an evaluator. Score the correctness of the generated response "
     "compared to the reference answer. Return ONLY JSON: 'score': <0-1>"
    ),

    ("human",
     "USER QUESTION:\n{question}\n\n"
     "GENERATED ANSWER:\n{answer}\n\n"
     "REFERENCE ANSWER:\n{reference}\n\n"
     "Return JSON only."
    )
])

def correctness(inputs: dict, outputs: dict, reference_outputs = None) -> float:
    msgs = correctness_prompt.format_messages(
        question = inputs["input"],
        answer = outputs["answer"],
        reference = reference_outputs["expected_output"]
    )
    res: MetricScore = judge.invoke(msgs)
    return res["score"]

In [38]:
groundness_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Evaluate how grounded the answer is in the retrieved documents.\n"
     "Use ONLY the docs, no world knowledge.\n"
     "Return ONLY JSON: 'score': <0-1>"
    ),

    ("human",
     "RETRIEVED DOCS:\n{docs}\n\n"
     "GENERATED ANSWER:\n{answer}"
    )
])

def groundness(inputs: dict, outputs: dict) -> float:
    docs = "\n\n".join(d for d in outputs["context"][0])
    msgs = groundness_prompt.format_messages(
        docs = docs,
        answer = outputs["answer"]
    )
    res: MetricScore = judge.invoke(msgs)
    return res["score"]

In [39]:
relevance_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Evaluate how well the answer addresses the user's question.\n"
     "Return ONLY JSON: 'score': <0-1>"
    ),

    ("human",
     "QUESTION:\n{question}\n\n"
     "ANSWER:\n{answer}"
    )
])

def relevance(inputs: dict, outputs: dict) -> float:
    msgs = relevance_prompt.format_messages(
        question = inputs["input"],
        answer = outputs["answer"]
    )
    res: MetricScore = judge.invoke(msgs)
    return res["score"]

In [40]:
retrieval_relevance_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Evaluate how relevant the retrieved documents are for the user's query.\n"
     "Return ONLY JSON: 'score': <0-1>"
    ),

    ("human",
     "QUESTION:\n{question}\n\n"
     "RETRIEVED DOCS:\n{docs}"
    )
])

def retreival_relevance(inputs: dict, outputs: dict) -> float:
    docs = "\n\n".join(d for d in outputs["context"][0])
    msgs = retrieval_relevance_prompt.format_messages(
        question = inputs["input"],
        docs = docs
    )
    res: MetricScore = judge.invoke(msgs)
    return res["score"]

In [41]:
coherence_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Evaluate clarity, structure, and coherence.\n"
     "Return ONLY JSON: 'score': <0-1>"
    ),

    ("human",
     "GENERATED ANSWER:\n{answer}"
    )
])

def coherence(inputs: dict, outputs: dict) -> float:
    msgs = coherence_prompt.format_messages(
        answer = outputs["answer"]
    )
    res: MetricScore = judge.invoke(msgs)
    return res["score"]

**RAGAS: LLM_Metrics**

In [42]:
from ragas import SingleTurnSample
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import NonLLMContextPrecisionWithReference, LLMContextPrecisionWithReference, NonLLMContextRecall, LLMContextRecall
from ragas.metrics.collections import RougeScore, BleuScore, ExactMatch, NonLLMStringSimilarity, DistanceMeasure


llm = ChatOpenAI(
    model = LLM,
    api_key = GROQ_API_KEY,
    base_url = BASE_URL,
    streaming = True
)

evaluator_llm = LangchainLLMWrapper(llm)

/tmp/ipykernel_6236/2883602223.py:14: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use the modern LLM providers instead: from ragas.llms.base import llm_factory; llm = llm_factory('gpt-4o-mini') or from ragas.llms.base import instructor_llm_factory; llm = instructor_llm_factory('openai', client=openai_client)
  evaluator_llm = LangchainLLMWrapper(llm)


In [43]:
context_recall_judge = LLMContextRecall(llm=evaluator_llm)
async def llm_context_recall(inputs:dict, outputs:dict, reference_outputs = None):
    """
    Measures how many of the relvant documents were successfully retreived.
    return : 0 - 1
    """
    example = SingleTurnSample(
        user_input = inputs['input'],
        response = outputs['answer'],
        reference = reference_outputs['expected_output'],
        retrieved_contexts = outputs['context'][0],
    )
    return await context_recall_judge.single_turn_ascore(example)

In [44]:
context_precision_judge = LLMContextPrecisionWithReference(llm=evaluator_llm)
async def llm_context_precision(inputs:dict, outputs:dict, reference_outputs = None):
    """
    A llm based methods to determine wheather a retreived context is relevant

    values used for measuring:
        user_input: str
        reference: str
        reference_contexts: list[str]
    """
    example = SingleTurnSample(
        user_input = inputs['input'],
        reference = reference_outputs['expected_output'],
        retrieved_contexts = outputs['context'][0],
    )

    return await context_precision_judge.single_turn_ascore(example)

**RAGAS: Non-LLM Metrics**

In [45]:
context_recall = NonLLMContextRecall()
async def non_llm_context_recall(inputs:dict, outputs:dict, example=None):
    sample = SingleTurnSample(
        retrieved_contexts = outputs['context'][0],
        reference_contexts = example.metadata['meta_data']['evidence']
    )
    return await context_recall.single_turn_ascore(sample)

In [46]:
context_precision = NonLLMContextPrecisionWithReference()
async def non_llm_context_precision(inputs:dict, outputs:dict, example=None):
    """
    A non-llm based methods to determine wheather a retreived context is relevant

    values used for measuring:
        retirieved_contexts: list[str]
        reference_contexts: list[str]
    """
    sample = SingleTurnSample(
        retrieved_contexts = outputs['context'][0],
        reference_contexts = example.metadata['meta_data']['evidence']
    )
    return await context_precision.single_turn_ascore(sample)


In [47]:
em_scorer = ExactMatch()
async def EM(inputs:dict, outputs:dict, reference_outputs = None):
    result = await em_scorer.ascore(
        reference = reference_outputs['expected_output'],
        response = outputs['answer']
    )

    return result.value

In [48]:
SS_scorer = NonLLMStringSimilarity(distance_measure=DistanceMeasure.LEVENSHTEIN)
async def String_Similarity(inputs:dict, outputs:dict, reference_outputs = None):
    result = await SS_scorer.ascore(
        reference = reference_outputs['expected_output'],
        response = outputs['answer']
    )

    return result.value

In [49]:
bleu_scorer = BleuScore()
async def BLUE(inputs:dict, outputs:dict, reference_outputs = None):
    result = await bleu_scorer.ascore(
        reference = reference_outputs['expected_output'],
        response = outputs['answer']
    )

    return result.value

In [50]:
rouge_scorer = RougeScore(rouge_type="rougeL", mode="fmeasure")
async def rougeL(inputs:dict, outputs:dict, reference_outputs = None):
    result = await rouge_scorer.ascore(
        reference = reference_outputs['expected_output'],
        response = outputs['answer']
    )

    return result.value

In [51]:
inputs = {
    "input":"Did the annotators agreed and how much?",
}
reference_outputs = {
    "expected_output":"""For event types and participant types, there was a moderate to substantial level of agreement using the Fleiss' Kappa. For coreference chain annotation, there was average agreement of 90.5%.
Moderate agreement of 0.64-0.68 Fleiss’ Kappa over event type labels, 0.77 Fleiss’ Kappa over participant labels, and good agreement of 90.5% over coreference information."""
}
outputs = {
    "answer":"For event types and participant types, there was a moderate to substantial level of agreement using the Fleiss' Kappa. For coreference chain annotation, there was average agreement of 90.5%.",
    "context": [
        [
            "In order to calculate inter-annotator agreement, a total of 30 stories from 6 scenarios were randomly chosen for parallel annotation by all 4 annotators after the first annotation phase. We checked the agreement on these data using Fleiss' Kappa BIBREF4 . The results are shown in Figure 4 and indicate moderate to substantial agreement BIBREF5 . Interestingly, if we calculated the Kappa only on the subset of cases that were annotated with script-specific event and participant labels by all annotators, results were better than those of the evaluation on all labeled instances (including also unrelated and related non-script events). This indicates one of the challenges of the annotation task: In many cases it is difficult to decide whether a particular event should be considered a central script event, or an event loosely related or unrelated to the script.",
            "For coreference chain annotation, we calculated the percentage of pairs which were annotated by at least 3 annotators (qualified majority vote) compared to the set of those pairs annotated by at least one person (see Figure 4 ). We take the result of 90.5% between annotators to be a good agreement.",
            "FLOAT SELECTED: Figure 4: Inter-annotator agreement statistics."
        ]
    ]
}
class Example:
    def __init__(self):
        self.metadata = {
            "meta_data": {
                "evidence": [
                    "For coreference chain annotation, we calculated the percentage of pairs which were annotated by at least 3 annotators (qualified majority vote) compared to the set of those pairs annotated by at least one person (see Figure 4 ). We take the result of 90.5% between annotators to be a good agreement.",
                    "FLOAT SELECTED: Figure 4: Inter-annotator agreement statistics."
                ]
            }
        }
e = Example()

In [52]:
await rougeL(inputs, outputs, reference_outputs)

0.6741573033707865

## Evaluation

**Get Your Generated Answers**

In [53]:
import json
QA_pairs = []

with open("qasper_mini_qa_generated.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        qa = json.loads(line)
        QA_pairs.append(qa)

In [ ]:
from langsmith import Client

client = Client(api_key = "", tracing_sampling_rate = 0)
dataset_name = "qasper_mini"

**Target Function**

In [55]:
async def target(inputs: dict) -> dict:
    global operation
    res = [qa["outputs"] for qa in QA_pairs if qa["input"] == inputs["input"]][0]
    if operation % 10 == 0:
        print(f"============================================== {operation}")
        
    return res

**Dummy Evaluator**

In [56]:
operation = 0
def dummy(inputs: dict, outputs:dict, example):
    global operation

    operation += 1
    
    
    return 0

**Dummy Evaluation**

In [ ]:
evaluation_results = await client.aevaluate(
    target,
    data=dataset_name,
    evaluators=[dummy],
    experiment_prefix="rag_evaluation",
    upload_results=False
)

In [1]:
# correctness, groundness, relevance, retreival_relevance, coherence, llm_context_recall, llm_context_precision, non_llm_context_recall, non_llm_context_precision, EM, String_Similarity, BLUE, rougeL

**Main Evaluation**

In [57]:
evaluation_results = await client.aevaluate(
    target,
    data=dataset_name,
    evaluators=[dummy, non_llm_context_recall, non_llm_context_precision, EM, String_Similarity, BLUE, rougeL],
    experiment_prefix="rag_evaluation",
    upload_results=True
)

View the evaluation results for experiment: 'rag_evaluation-9376914c' at:
https://smith.langchain.com/o/21586c76-7294-4a45-a960-11a6c111eae6/datasets/ddc8f40c-4e1b-491b-bc1d-20067a2c4956/compare?selectedSessions=84a0e250-fe0a-44af-9d90-151f4eb22a05




0it [00:00, ?it/s]

============================================== 0
============================================== 10
============================================== 20
============================================== 30
============================================== 40
============================================== 50
============================================== 60
============================================== 70
============================================== 80
============================================== 90
============================================== 100
============================================== 110
============================================== 120


In [58]:
E = evaluation_results.to_pandas()

In [ ]:
E.describe()

,feedback.dummy,feedback.non_llm_context_recall,feedback.non_llm_context_precision,feedback.EM,feedback.String_Similarity,feedback.BLUE,feedback.rougeL,execution_time
count,124.0,124.000000,124.000000,124.0,124.000000,124.000000,124.000000,124.000000
mean,0.0,0.016398,0.009969,0.0,0.171431,0.021422,0.081435,0.000798
std,0.0,0.083994,0.053266,0.0,0.088960,0.037539,0.092744,0.000421
min,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000349
25%,0.0,0.000000,0.000000,0.0,0.115307,0.000000,0.000000,0.000478
50%,0.0,0.000000,0.000000,0.0,0.173049,0.010989,0.066667,0.000648
75%,0.0,0.000000,0.000000,0.0,0.231230,0.028311,0.127153,0.000997
max,0.0,0.500000,0.333333,0.0,0.468750,0.234624,0.400000,0.002344


In [60]:
E.to_csv("Non_LLM_Evaluation_results.csv")